# Section 1 — Your First Agent with the Claude Agent SDK

**Course:** Build Production AI Agents with the Claude Agent SDK

This single notebook walks you from an empty environment to a working agent, one concept at a time.

## Roadmap

| Part | Topic | What you'll learn |
|------|-------|-------------------|
| 1 | Setup | Install the SDK, store your API key safely, pick a model |
| 2 | Your first `query()` call | Send a prompt, give the agent tools, watch the message stream |
| 3 | Reading the message stream | The 4 message types, content blocks, and the clean production pattern |
| 4 | Build a File Explorer Agent | Combine `Glob` + `Read` to explore a real project |
| 5 | The prompt is your control lever | Same code, new prompt, new behaviour |
| 6 | Try it yourself + Summary | An exercise, a cheat sheet and troubleshooting tips |

**Run the cells from top to bottom.** Every later part builds on the setup in Part 1.

> **Prerequisites:** Python 3.10+, Jupyter or VS Code, and an Anthropic API key.

---
# Part 1 — Setup

## 1.1 Get an Anthropic API key

The agent talks to Claude through the Anthropic API, so it needs a key.

1. Sign up or log in at [platform.claude.com](https://platform.claude.com/)
2. Open **API Keys** in the sidebar and click **Create Key**
3. Give it a descriptive name (e.g. *Claude Agent SDK Learning*)
4. **Copy the key immediately.** Anthropic shows it only once.

> ⚠️ Treat the key like a password. Never paste it into a notebook cell or share it in a screenshot.

## 1.2 Store the key in a `.env` file

Put the key in a local **`.env`** file so it never ends up inside the notebook:

1. In the **same folder as this notebook**, create a file named exactly `.env`
2. Add one line: `ANTHROPIC_API_KEY=your-api-key-here`
3. Replace the placeholder with your real key and save
4. Make sure `.env` is listed in your `.gitignore`

> **Why a `.env` file?** A key typed into a code cell is saved inside the `.ipynb` file, and anyone you share the notebook with can read it. `python-dotenv` loads the key from `.env` at run time, so the notebook stays safe to share.

## 1.3 Install the packages

We pin the SDK version so every example behaves exactly as described here. To try newer features, remove the `==0.2.93` pin (newer versions may behave differently).

In [ ]:
# Install the Claude Agent SDK and python-dotenv (-q keeps pip output quiet).
%pip install claude-agent-sdk==0.2.140 python-dotenv -q

## 1.4 Load the API key

`load_dotenv()` reads `.env` and copies `ANTHROPIC_API_KEY` into `os.environ`. The SDK looks for it there automatically, so we never pass the key around ourselves.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print("API key loaded:", os.environ.get("ANTHROPIC_API_KEY") is not None)

> If this prints `False`, check that `.env` is in the same folder as this notebook, is named exactly `.env`, and contains `ANTHROPIC_API_KEY=...`. Then re-run the cell.

## 1.5 Run async agent code from a notebook

`query()` is **asynchronous** and launches the Claude Code CLI as a subprocess. That causes one wrinkle on Windows: Jupyter's default event loop (`SelectorEventLoop`) cannot spawn subprocesses, and `query()` fails with `NotImplementedError`.

The helper below runs any `async def` on its own thread with a Proactor-capable loop. It works on every platform and returns whatever your function returns. We define it **once** and reuse it in every later part.

In [ ]:
import asyncio
import sys
import threading


def run_sync(coro_fn):
    """Run an `async def` function to completion and return its result.

    Windows: Jupyter's kernel loop is a SelectorEventLoop, which cannot spawn
    subprocesses. The Claude Agent SDK launches the Claude Code CLI as a
    subprocess, so we run on a separate thread with its own Proactor loop.
    On macOS/Linux the policy change is a no-op.
    """
    outcome = {}

    def runner():
        if sys.platform == "win32":
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        try:
            outcome["value"] = asyncio.run(coro_fn())
        except Exception as exc:
            outcome["error"] = exc

    thread = threading.Thread(target=runner)
    thread.start()
    thread.join()

    if "error" in outcome:
        raise outcome["error"]
    return outcome.get("value")

## 1.6 Choose a model

One variable controls which Claude model every agent in this notebook uses. We default to **Haiku**, which is fast and cheap while you are learning.

| Model | Trade-off |
|-------|-----------|
| `claude-haiku-5-5` | Fastest, lowest cost (default here) |
| `claude-sonnet-5-5` | More capable, higher cost |
| `claude-opus-5-5` | Most capable, highest cost |

Latest models: <https://platform.claude.com/docs/en/about-claude/models/overview>

In [ ]:
MODEL_NAME = "claude-haiku-5-5"   # change to "claude-sonnet-5-5" or "claude-opus-5-5" if you like

## 1.7 Verify the setup

In [ ]:
import claude_agent_sdk

print("SDK installed successfully!")
print("API key set:", os.environ.get("ANTHROPIC_API_KEY") is not None)
print("Model:", MODEL_NAME)

✅ **Environment ready.** This is the only setup you need in this notebook. Everything from here on just works.

---
# Part 2 — Your First `query()` Call

`query()` is the heart of the Claude Agent SDK. You give it two things:

| Argument | What it is |
|----------|-----------|
| `prompt` | The goal, in plain English |
| `options` | A `ClaudeAgentOptions` object: which tools the agent may use, which model, and more |

It returns an **async stream of messages**. Each message is something the agent did: a tool it chose to call, the result of that call, or the final answer. You consume the stream with `async for`.

## Tools: what the agent is allowed to do

An agent with no tools can only talk. Tools let it act. `allowed_tools` is the list of tools Claude may use without asking first.

| Tool | What it does |
|------|--------------|
| `Bash` | Run shell commands |
| `Glob` | Find files by pattern (e.g. `**/*.py`) |
| `Read` | Read a file |
| `Grep` | Search inside files |

## Example 2.1 — "What files are in this directory?"

We give Claude `Bash` and `Glob` and a goal. **We do not say how to answer.** Claude decides which tool to use and in what order. That is what makes it an agent.

We collect every message into a list so we can study them in Part 3 without calling the API again.

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions

#agentic loop
# prompt -> What files are in this directory
# plan
# thinkingblock -> i need to get the list of folder and files from the currnet directory
# current directory
# prepare the command for listing all the files 
# get list of files from the directory
# prepsre the output message 

#response = query(
#        prompt="What files are in this directory?",
#        options=ClaudeAgentOptions(
#            allowed_tools=["Bash", "Glob"],
#            model=MODEL_NAME,
#        ),
#    )

async def first_agent():
    messages = []
    async for message in query(
        prompt="What files are in this directory?",
        options=ClaudeAgentOptions(
            tools=["Bash", "Glob"],
            allowed_tools=["Bash"],
            model=MODEL_NAME,
        ),
    ):
        print(message)          # show everything, live
        messages.append(message)  # keep it for Part 3
    return messages


first_run_messages = run_sync(first_agent)

## What just happened?

1. **Claude received the prompt and the list of allowed tools** and entered its internal *agent loop*.
2. **Claude chose a tool.** It decided on its own to call `Bash` or `Glob` to inspect the file system. That decision was Claude's, not yours.
3. **The tool ran and returned a result.** Claude read the output and decided whether it had enough information.
4. **Claude left the loop and returned a final answer**, which arrives as the last message in the stream: a `ResultMessage`.
5. **You looped over the stream** with `async for`.

The raw output looks busy. In Part 3 we name every object in it.

---
# Part 3 — Reading Agent Messages & the Result Stream

> **Key insight:** The SDK is the only thing that writes to your stream. Claude returns JSON over HTTP, and the SDK builds every Python object you see. The difference between message types is about *who decided the content*, not who delivered it.

## 3.1 The four top-level message types

| Message type | Content decided by | When it appears | Key fields |
|--------------|--------------------|-----------------|-----------|
| `SystemMessage` | SDK | Before Claude sees anything | `subtype`, `data` (includes `session_id`) |
| `AssistantMessage` | **Claude** (packaged by the SDK) | Every Claude response | `content` (a list of blocks) |
| `UserMessage` | SDK | After each tool runs | `content` (a list of blocks) |
| `ResultMessage` | SDK | Once, when the loop is done | `result`, `total_cost_usd`, `num_turns`, `duration_ms` |

> ⚠️ There is **no** `ToolUseMessage` or `ToolResultMessage` in the Python SDK. Tool activity lives *inside* `AssistantMessage` and `UserMessage` as **content blocks**.

## 3.2 Content blocks

**Inside `AssistantMessage.content`:**

| Block | Key fields | Meaning |
|-------|-----------|---------|
| `TextBlock` | `block.text` | Claude's plain-text output |
| `ToolUseBlock` | `block.name`, `block.input` | Claude calling a tool |
| `ThinkingBlock` | `block.thinking` | Claude's extended reasoning (thinking mode) |

**Inside `UserMessage.content`:**

| Block | Key fields | Meaning |
|-------|-----------|---------|
| `ToolResultBlock` | `block.content` | What a tool returned |

## 3.3 The two-level pattern

```
isinstance() on the outer message
  └── if AssistantMessage or UserMessage:
        loop over message.content
        └── isinstance() on each block
```

> **Your prompt is never in the stream.** The SDK sends your `prompt=` to Claude internally. The first message you see is always the `SystemMessage`.

## Example 3.1 — Decode the stream we already captured

We reuse `first_run_messages` from Part 2, so this cell makes **no new API call**. Compare each line below with the raw output you saw earlier.

In [ ]:
from claude_agent_sdk import (
    SystemMessage,
    AssistantMessage,
    UserMessage,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
    ToolResultBlock,
    ThinkingBlock,
)


def describe(message):
    """Print a one-line summary for every message and block in the stream."""
    if isinstance(message, SystemMessage):
        # SDK-generated, before Claude sees anything
        print(f"[SYSTEM]       SDK init: subtype={message.subtype}")

    elif isinstance(message, AssistantMessage):
        # Claude decided the content, the SDK packaged it
        for block in message.content:
            if isinstance(block, TextBlock):
                print(f"[TEXT]         {block.text[:80]}")
            elif isinstance(block, ToolUseBlock):
                print(f"[TOOL USE]     tool={block.name}  input={block.input}")
            elif isinstance(block, ThinkingBlock):
                print(f"[THINKING]     {block.thinking[:60]}...")

    elif isinstance(message, UserMessage):
        # SDK-generated: the tool result fed back to Claude
        for block in message.content:
            if isinstance(block, ToolResultBlock):
                print(f"[TOOL RESULT]  {str(block.content)[:80]}")

    elif isinstance(message, ResultMessage):
        # SDK-generated: the loop is complete (appears once)
        print(f"[RESULT]       {message.result[:80]}")
        print(f"               cost=${message.total_cost_usd:.4f}  "
              f"turns={message.num_turns}  "
              f"duration={message.duration_ms}ms")


for msg in first_run_messages:
    describe(msg)

# [:80] only keeps the notebook output tidy. In production, read the full content.

## 3.4 The clean production pattern

In most real agents (a CI/CD pipeline, a backend service, a user-facing app) you only need the **final answer**. The pattern below is the default for the rest of the course.

### Why `hasattr(message, "result")` and not `TextBlock`?

- An `AssistantMessage` with a `TextBlock` can appear **many times** in one run, including as commentary between tool calls.
- A `ResultMessage` appears **exactly once**, at the very end.
- Its `.result` is the same text as the last `TextBlock`, **plus** run metadata:
  - `total_cost_usd`: cost of the full run
  - `num_turns`: how many tool-call rounds happened
  - `duration_ms`: total elapsed time
  - `is_error`: whether the run ended in an error

`hasattr(message, "result")` waits for the SDK's signal that the task is fully complete. The explicit alternative, `isinstance(message, ResultMessage)`, is equally correct.

## Example 3.2 — Only the final answer

In [ ]:
async def final_answer_only():
    async for message in query(
        prompt="What files are in this directory?",
        options=ClaudeAgentOptions(
            allowed_tools=["Bash", "Glob"],
            model=MODEL_NAME,
        ),
    ):
        if hasattr(message, "result"):   # only ResultMessage has .result
            print(message.result)


run_sync(final_answer_only)

## 3.5 When would you read content blocks?

`hasattr(message, "result")` is the right default. Read content blocks when you need:

| Scenario | What you read |
|----------|---------------|
| 🔍 **Debugging** | `ToolUseBlock` (tool name and input) and `ToolResultBlock` (what came back) |
| 📺 **Real-time UI progress** | `ToolUseBlock` arrives, so show "Claude is searching…" or "Running tests…" |
| 📋 **Audit logging** (Section 5: Hooks) | Log every `ToolUseBlock` to a file for compliance |

```python
# Debugging snippet
elif isinstance(message, AssistantMessage):
    for block in message.content:
        if isinstance(block, ToolUseBlock):
            print(f"Tool called: {block.name}")
            print(f"Input:       {block.input}")

elif isinstance(message, UserMessage):
    for block in message.content:
        if isinstance(block, ToolResultBlock):
            print(f"Result:      {block.content}")
```

---
# Part 4 — Build a File Explorer Agent

Time to put it all together. We build an agent that **autonomously explores a project directory, reads every file, and writes a summary report**. You describe the goal in plain English and the agent does the rest.

## 4.1 The local filesystem

This notebook runs against **your machine's real filesystem**, and the working directory is the folder holding this notebook.

- `sample_project/` (next to the notebook) is where we build the demo project.
- We create the sample files **in code**, so every run starts clean and predictable.
- Agent tools (`Read`, `Glob`, `Grep`) operate on real local files.
- Files persist after the kernel stops. Delete the folder when you are done.

## 4.2 Create the sample project

Creating files programmatically gives us three things:

1. **Full control:** we decide exactly what exists.
2. **Reproducibility:** everyone gets identical files.
3. **Predictability:** we know what a correct summary should say.

The project is a tiny Python library: a README, an auth module, a utils module and a test file.

In [ ]:
import os

SAMPLE_FILES = {
    "sample_project/README.md": (
        "# Sample Project\n"
        "This is a demo project for the File Explorer Agent.\n"
        "It contains an authentication module, utility functions, and tests.\n"
    ),
    "sample_project/src/auth.py": (
        "# Authentication module\n\n"
        "def login(user, password):\n"
        '    """Authenticate a user with username and password."""\n'
        "    return user == 'admin' and password == 'secret'\n\n"
        "def logout(user):\n"
        '    """Log out the specified user."""\n'
        "    return True\n\n"
        "def reset_password(user, new_password):\n"
        '    """Reset the password for the specified user."""\n'
        "    return True\n"
    ),
    "sample_project/src/utils.py": (
        "# Utility functions\n\n"
        "def format_date(date):\n"
        '    """Format a date object as YYYY-MM-DD string."""\n'
        "    return date.strftime('%Y-%m-%d')\n\n"
        "def format_currency(amount):\n"
        '    """Format a number as a currency string."""\n'
        "    return f'${amount:.2f}'\n\n"
        "def validate_email(email):\n"
        '    """Check if an email address is valid."""\n'
        "    return '@' in email\n"
    ),
    "sample_project/tests/test_auth.py": (
        "# Tests for auth module\n\n"
        "def test_login():\n"
        '    """Test the login function."""\n'
        "    assert login('admin', 'secret') == True\n\n"
        "def test_logout():\n"
        '    """Test the logout function."""\n'
        "    assert logout('admin') == True\n"
    ),
}

for path, content in SAMPLE_FILES.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)

print("Sample project created successfully.")

## 4.3 Verify what was created

Before running the agent, print exactly what it is going to read. Then you can check its summary against the truth. **No black boxes.**

In [ ]:
print("Project structure:")
print("=" * 40)
for root, dirs, files in os.walk("sample_project"):
    level = root.replace("sample_project", "").count(os.sep)
    print(f"{'  ' * level}{os.path.basename(root)}/")
    for file in files:
        print(f"{'  ' * (level + 1)}{file}")

print("\nFile contents:")
print("=" * 40)
for filepath in SAMPLE_FILES:
    print(f"\n--- {filepath} ---")
    with open(filepath) as f:
        print(f.read())

## 4.4 The File Explorer Agent

**Two tools:**

- **`Glob`** finds files by pattern, like a powerful `find` (e.g. `**/*.py` finds every Python file).
- **`Read`** opens and reads any file in the working directory.

**The pattern:** give the agent a goal as a numbered prompt. You never tell it to use `Glob` before `Read`. It works out the order itself. We extract the final answer with the `hasattr(message, "result")` pattern from Part 3.

Because we will reuse this agent with different prompts in Part 5, we wrap it in a small function that takes the prompt as its argument.

In [ ]:
# File agent -> work with my files 

def explore(prompt):
    """Run a read-only File Explorer agent (Glob + Read) and print its final report."""

    async def main():
        async for message in query(
            prompt=prompt,
            options=ClaudeAgentOptions(
                allowed_tools=["Glob", "Read"],
                model=MODEL_NAME,
            ),
        ):
            if hasattr(message, "result"):
                print(message.result)

    run_sync(main)


explore("""
Explore the project at sample_project.
1. List all files and folders in the project
2. Read each file
3. For each file, write a one sentence summary of what it contains
4. Present the results as a clean, readable report
""")

None


KeyboardInterrupt: 

## What did the agent do, and what did *you* not do?

- You did **not** tell it to use `Glob` before `Read`.
- You did **not** write a loop to open each file.
- You described a **goal**, and the agent figured out the tool sequence.

That is the **agent loop** running on a real task:

1. Receive the goal (your prompt)
2. Decide which tool to call first
3. Call the tool and get the result
4. Use the result to decide the next step
5. Repeat until the goal is met
6. Return the final answer as a `ResultMessage`

**Expected output** (wording varies, since Claude writes it fresh each time):

```
Project Summary — sample_project
==========================================

README.md
  A project description explaining this is a demo project containing
  authentication, utility, and test modules.

src/auth.py
  An authentication module with login, logout, and reset_password
  functions for managing user access.

src/utils.py
  A utility module with functions for formatting dates, currency values,
  and validating email addresses.

tests/test_auth.py
  A test module containing tests for the login and logout functions
  in the authentication module.
```

---
# Part 5 — The Prompt Is Your Control Lever

The prompt is the most powerful control you have when building agents. **Change the prompt and you change the agent**, without touching any other code.

Below we call the **same** `explore()` function with the **same** tools and only swap the prompt. Instead of file summaries we ask for a function inventory.

Same agent. Different goal. Different output.

In [ ]:
explore("""
Explore the project at sample_project.
1. Find all Python files only
2. For each Python file, list all the function names defined in it
3. Present the results as a clean report
""")

**Expected output** (approximate):

```
Python Functions by File
========================

src/auth.py
  - login(user, password)
  - logout(user)
  - reset_password(user, new_password)

src/utils.py
  - format_date(date)
  - format_currency(amount)
  - validate_email(email)

tests/test_auth.py
  - test_login()
  - test_logout()
```

---
# Part 6 — Try It Yourself, Summary & Troubleshooting

## Exercise (optional)

Reuse `explore()` and write a prompt that makes the agent do something new. A few ideas:

1. Find every function that has **no docstring**, or confirm that all of them have one.
2. List which functions in `src/` have **no matching test** in `tests/`.
3. Spot a bug in the test file. *(Hint: look at what `test_auth.py` is missing at the top.)*

Write your prompt in the cell below.

In [ ]:
# Your turn: write a prompt and run it with explore()

explore("""
# Type your prompt here
""")

## Summary

### What we built

A **File Explorer Agent** that discovers files with `Glob`, reads them with `Read`, produces a structured report, and can be redirected to a different task by changing only the prompt.

### Concept cheat sheet

| Concept | What it is |
|---------|------------|
| `.env` + `python-dotenv` | Keeps the API key out of the notebook and out of git |
| `run_sync()` | One helper that runs async agent code in a notebook on any OS |
| `MODEL_NAME` | One variable controlling which Claude model every agent uses |
| `query(prompt, options)` | Starts the agent and returns an async stream of messages |
| `allowed_tools` | Which tools the agent may use (`Bash`, `Glob`, `Read`, …) |
| Agent loop | Claude picks a tool → SDK runs it → result goes back to Claude → repeat until done |
| 4 message types | `SystemMessage`, `AssistantMessage`, `UserMessage`, `ResultMessage` |
| Content blocks | `TextBlock`, `ToolUseBlock`, `ThinkingBlock`, `ToolResultBlock` live *inside* messages |
| `hasattr(message, "result")` | The clean pattern for getting the final answer (fires once) |
| Verification pattern | Print the inputs first so you can check the agent's output |
| Prompt as control lever | One prompt change gives completely different agent behaviour |

## Troubleshooting

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| `API key loaded: False` | `.env` missing, misnamed or in another folder | Create `.env` next to the notebook with `ANTHROPIC_API_KEY=...`, then re-run Part 1.4 |
| Authentication error when running an agent | Wrong or revoked key | Create a new key and update `.env`. Restart the kernel so it is reloaded. |
| `NotImplementedError` on Windows | Jupyter's event loop cannot spawn subprocesses | Run agents through `run_sync()` and do not `async for` directly in a cell |
| `ModuleNotFoundError: claude_agent_sdk` | Install cell not run, or kernel not restarted | Re-run Part 1.3, then restart the kernel if needed |
| Output wording differs from the examples | Claude writes fresh text every run | Expected. Check the *content* against the files you created. |
| Kernel restarted and variables are gone | Notebook state is in memory | Re-run from Part 1 (`run_sync`, `MODEL_NAME`, imports) |

### What's next

**Section 3 — Built-in Tools Deep Dive:** a deeper look at the built-in tools the agent can use.

---
*Great work. You built your first complete agent.* 🎉